# Day 2 — Lesson 95 Challenge: **Python → C++ Code Converter & Benchmark** (Frontier + Open‑Source)

This notebook conducts the **commercial challenge** from Lesson 95: build a product that converts **Python code to C++** for performance, **two ways**:

- **Frontier model path**: use an API model (OpenAI or Anthropic) if you have keys.
- **Open‑source path**: run an HF Transformers **code LLM** locally (default: `Qwen/Qwen2.5-Coder-7B-Instruct`) with **4‑bit quantization** on a **T4 GPU**.

It then **compiles** the C++ (g++), **executes** it on test cases, and **benchmarks** against the original Python implementation. A simple **Gradio UI** ties it all together.

> **Designed for Google Colab (T4) and local Jupyter.** If you have a T4 16GB, keep default 7B models and 4‑bit to stay within memory.

## 0) Environment check

In [1]:
import os, sys, platform, subprocess, shutil
print("Python:", sys.version)
print("Platform:", platform.platform())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
try:
    import torch
    print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch not installed yet", e)

Python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
Platform: Linux-6.1.123+-x86_64-with-glibc2.35
CUDA_VISIBLE_DEVICES: None
Torch: 2.8.0+cu126 | CUDA: True | GPUs: 1
GPU: NVIDIA A100-SXM4-40GB


## 1) Installs
- **Transformers stack** for open‑source LLM (4‑bit via bitsandbytes)
- **Gradio** for UI
- **OpenAI / Anthropic** (optional) for frontier path

In [2]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece gradio pandas
!pip install -q -U openai==1.* anthropic==0.*

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 127.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.2 which is incompatible.
dask-cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.2 which is incompatible.
cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 930.8/930.8 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.0/

## 2) Imports & device

In [3]:
import os, re, time, json, tempfile, textwrap, pathlib, subprocess, shlex
from typing import List, Dict, Any, Optional, Tuple

import pandas as pd
import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TextIteratorStreamer

# Optional clients
try:
    from openai import OpenAI
except Exception:
    OpenAI = None
try:
    import anthropic
except Exception:
    anthropic = None

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 3) (Optional) Google Drive (Colab) mount

In [4]:
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    print("Drive mounted at /content/drive")
except Exception:
    print("Not running in Colab or Drive not available.")

Mounted at /content/drive
Drive mounted at /content/drive


## 4) Auth & defaults

**Frontier models** (set one or both):
- `OPENAI_API_KEY` for OpenAI (models like `gpt-4o-mini`, `o4-mini`, `gpt-4.1-mini`)
- `ANTHROPIC_API_KEY` for Anthropic (e.g., `claude-3-5-sonnet-20240620` or newer)

**Open‑source model** (default fits on T4 with 4‑bit):  
- `Qwen/Qwen2.5-Coder-7B-Instruct`

In [54]:
# --- API keys (set env vars if you plan to use the frontier path) ---
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", None)
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", None)

openai_client = None
if OPENAI_API_KEY and OpenAI is not None:
    try:
        openai_client = OpenAI(api_key=OPENAI_API_KEY)
        print("✅ OpenAI client ready")
    except Exception as e:
        print("⚠️ OpenAI client init failed:", e)

anthropic_client = None
if ANTHROPIC_API_KEY and anthropic is not None:
    try:
        anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        print("✅ Anthropic client ready")
    except Exception as e:
        print("⚠️ Anthropic client init failed:", e)

# --- Open-source defaults ---
OS_DEFAULT = "Qwen/Qwen2.5-Coder-7B-Instruct"
OS_CHOICES = [
    "Qwen/Qwen2.5-Coder-7B-Instruct",
    "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct",
    "HuggingFaceH4/zephyr-7b-beta",
    "microsoft/Phi-3-mini-4k-instruct",  # fast/generalist fallback
]

✅ OpenAI client ready
✅ Anthropic client ready


## 5) Load open‑source code LLM (4‑bit on GPU when available)

In [17]:
_tok = None
_mdl = None

def load_os_model(model_id: str):
    global _tok, _mdl
    if _mdl is not None and getattr(_mdl, "name_or_path", None) == model_id:
        return _tok, _mdl
    print(f"Loading open-source model: {model_id}")
    kwargs = {}
    if DEVICE == "cuda":
        try:
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
            )
            kwargs["device_map"] = "auto"
            kwargs["torch_dtype"] = torch.bfloat16
        except Exception as e:
            print("4-bit load failed, falling back:", e)
            kwargs["device_map"] = "auto"
    _tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    if _tok.pad_token is None:
        _tok.pad_token = _tok.eos_token
    _mdl = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    return _tok, _mdl

# Warm-load default lazily during first call to save RAM/time.

## 6) Conversion prompts

We instruct the model to return **only a C++ function** with an exact signature.  
Default demo task uses a single-argument function for easy auto‑benchmarking:

- **Python signature**: `def sum_squares(n: int) -> int:`  
- **C++ signature**: `long long sum_squares(long long n);`

In [18]:
SYSTEM_FRONTIER = (
    "You convert Python functions to high-performance C++17. "
    "Return only compilable C++ code. "
    "DO NOT include a main() function. "
    "Preserve exact function name and signature requested. "
    "Avoid external deps; only standard headers. "
    "Use simple types and loops; no I/O except function signature; no explanations."
)

SYSTEM_OPEN_SOURCE = SYSTEM_FRONTIER

def make_user_prompt(py_code: str, cpp_signature: str, perf_hints: str = "") -> str:
    return f"""    Convert the Python function below to C++17.

REQUIRED C++ SIGNATURE (exact): {cpp_signature}

Rules:
- Include necessary standard headers (e.g., <cstdint>, <vector>, <cmath>) only if needed.
- No main() function.
- No comments, no explanations.
- Deterministic, side-effect free. No dynamic allocation unless necessary.
- Optimize for speed (-O3). Prefer preallocation and simple loops.
{perf_hints}

### Python function
{py_code}
"""

## 7) Frontier generators (OpenAI / Anthropic)

In [55]:
def generate_cpp_frontier(provider: str, model: str, py_code: str, cpp_signature: str, perf_hints: str = "") -> str:
    print(f"--- Generating C++ with Frontier model: {provider}/{model} ---")
    prompt = make_user_prompt(py_code, cpp_signature, perf_hints)
    try:
        if provider == "OpenAI":
            if openai_client is None:
                raise RuntimeError("OpenAI client not configured. Please set OPENAI_API_KEY.")
            # Prefer the Responses API for generality
            try:
                resp = openai_client.responses.create(
                    model=model,
                    input=[
                        {"role": "system", "content": SYSTEM_FRONTIER},
                        {"role": "user", "content": prompt},
                    ],
                    temperature=0.2,
                    max_output_tokens=2048,
                )
                text = resp.output_text
                print("OpenAI API call successful.")
            except Exception as e:
                print(f"OpenAI Responses API failed, trying Chat Completions: {e}")
                # Fallback to Chat Completions (older)
                chat = openai_client.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": SYSTEM_FRONTIER},
                        {"role": "user", "content": prompt},
                    ],
                    temperature=0.2,
                    max_tokens=2048,
                )
                text = chat.choices[0].message.content
                print("OpenAI Chat Completions API call successful.")
            return text.strip()
        elif provider == "Anthropic":
            if anthropic_client is None:
                raise RuntimeError("Anthropic client not configured. Please set ANTHROPIC_API_KEY.")
            msg = anthropic_client.messages.create(
                model=model,
                max_tokens=2048,
                temperature=0.2,
                system=SYSTEM_FRONTIER,
                messages=[{"role": "user", "content": prompt}],
            )
            # Messages API returns content as a list of blocks
            parts = []
            for blk in msg.content:
                if getattr(blk, "type", "") == "text":
                    parts.append(blk.text)
            text = "\n".join(parts).strip()
            print("Anthropic API call successful.")
            return text
        else:
            raise ValueError(f"Unknown provider: {provider}")
    except Exception as e:
        error_message = f"An error occurred during Frontier API call: {type(e).__name__}: {e}\nTraceback:\n{traceback.format_exc()}"
        print(error_message) # Print to console for debugging
        print("--- Frontier generation failed ---")
        return f"Error: {type(e).__name__}: {e}"

import traceback

## 8) Open‑source generator (Transformers)

In [56]:
def generate_cpp_open_source(model_id: str, py_code: str, cpp_signature: str, perf_hints: str = "", temperature: float = 0.2, top_p: float = 0.95) -> str:
    tok, mdl = load_os_model(model_id)
    prompt = make_user_prompt(py_code, cpp_signature, perf_hints)
    messages = [
        {"role": "system", "content": SYSTEM_OPEN_SOURCE},
        {"role": "user", "content": prompt},
    ]
    input_ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
    if DEVICE == "cuda":
        input_ids = input_ids.to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(
            input_ids=input_ids,
            max_new_tokens=1536,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tok.eos_token_id,
        )
    gen = out[0, input_ids.shape[1]:]
    text = tok.decode(gen, skip_special_tokens=True).strip()
    # Strip possible fences
    text = re.sub(r"^```(?:cpp|c\+\+)?\n|\n```$", "", text, flags=re.IGNORECASE).strip()
    return text

## 9) C++ build & benchmark harness

We compile the generated function into a small program that **reads one integer** from `stdin`, calls the function, and prints the result.  
This supports the default demo task **`sum_squares(n)`**. You can adapt the template for other signatures.

In [57]:
TEMPLATE = r"""    #include <bits/stdc++.h>
using namespace std;
// === BEGIN MODEL FUNCTION ===
{model_function}
// === END MODEL FUNCTION ===

int main() {{
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    long long n;
    if(!(cin >> n)) return 1;
    auto ans = {func_name}(n);
    cout << ans << "\n";
    return 0;
}}
"""

def sanitize_cpp(code: str) -> str:
    # Remove accidental main() if model added it
    code = re.sub(r"\n\s*int\s+main\s*\([^)]*\)\s*\{[\s\S]*?\}\s*", "\n", code)
    # Remove markdown fences
    code = re.sub(r"^```(?:cpp|c\+\+)?\n|\n```$", "", code.strip(), flags=re.IGNORECASE)
    return code.strip()

def write_and_build(cpp_body: str, func_name: str, out_dir: Optional[str] = None) -> Tuple[str, str, str]:
    cpp_body = sanitize_cpp(cpp_body)
    tmp = out_dir or tempfile.mkdtemp()
    src = os.path.join(tmp, "model_func.cpp")
    exe = os.path.join(tmp, "a.out")
    full = TEMPLATE.format(model_function=cpp_body, func_name=func_name)
    with open(src, "w", encoding="utf-8") as f:
        f.write(full)
    # Build
    cmd = ["g++", "-O3", "-std=c++17", src, "-o", exe]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    return src, exe, (proc.stdout + "\n" + proc.stderr)

def run_binary(exe: str, n: int, timeout: float = 60.0) -> Tuple[Optional[int], float, str]:
    start = time.perf_counter()
    try:
        proc = subprocess.run([exe], input=f"{n}\n", text=True, capture_output=True, timeout=timeout)
        elapsed = time.perf_counter() - start
        if proc.returncode != 0:
            return None, elapsed, (proc.stdout + "\n" + proc.stderr)
        out = proc.stdout.strip()
        # Expect integer result
        val = int(out)
        return val, elapsed, ""
    except Exception as e:
        return None, time.perf_counter() - start, str(e)

## 10) Python reference runner & benchmark

In [58]:
def exec_python_function(py_code: str, func_name: str):
    scope = {}
    exec(py_code, scope)
    if func_name not in scope:
        raise ValueError(f"Function {func_name} not found after exec")
    return scope[func_name]

def bench_python(fn, n: int, repeats: int = 1) -> Tuple[int, float]:
    # Expect integer output
    start = time.perf_counter()
    out = None
    for _ in range(repeats):
        out = fn(n)
    elapsed = time.perf_counter() - start
    return int(out), elapsed

## 11) Default demo: **sum_squares**

Pure-Python loop is intentionally slow; C++ loop should be much faster.

In [59]:
DEFAULT_PY = """    def sum_squares(n: int) -> int:
    s = 0
    for i in range(1, n+1):
        s += i*i
    return s
"""
DEFAULT_FUNC = "sum_squares"
DEFAULT_CPP_SIG = "long long sum_squares(long long n);"
DEFAULT_TESTS = [1000, 2000000, 2500000]  # keep <= ~2.5e6 to stay within 64-bit range  # you can adjust if too slow/fast

## 12) Gradio UI

In [60]:
def do_build_and_bench(cpp_code, py_code, func_name, tests_csv):
    print("--- Starting build and bench ---")
    try:
        print(f"Tests CSV: {tests_csv}")
        tests = [int(x) for x in re.split(r"[\s,]+", tests_csv.strip()) if x.strip()]
        print(f"Parsed tests: {tests}")
        # Build
        print("Building C++ code...")
        src, exe, build_log = write_and_build(cpp_code, func_name)
        print(f"Build log:\n{build_log}")
        print(f"Executable path: {exe}")
        # Python ref
        print("Executing Python reference...")
        py_fn = exec_python_function(py_code, func_name)
        print("Python function executed.")
        rows = []
        for n in tests:
            print(f"Running test for n={n}")
            # Python
            py_out, py_sec = bench_python(py_fn, n, repeats=1)
            print(f"Python output: {py_out}, time: {py_sec:.6f}s")
            # C++
            cpp_out, cpp_sec, cpp_log = run_binary(exe, n)
            print(f"C++ output: {cpp_out}, time: {cpp_sec:.6f}s, log: {cpp_log}")
            ok = (cpp_out == py_out)
            rows.append({
                "n": n,
                "py_out": py_out,
                "cpp_out": cpp_out if cpp_out is not None else "ERR",
                "correct": bool(ok),
                "python_time_s": round(py_sec, 6),
                "cpp_time_s": round(cpp_sec, 6) if cpp_out is not None else None,
                "speedup_x": round((py_sec / cpp_sec), 2) if (cpp_out is not None and cpp_sec > 0) else None,
            })
        df = pd.DataFrame(rows)
        print("Benchmark completed successfully.")
        # Save artifacts
        tmp = tempfile.mkdtemp()
        cpp_path = os.path.join(tmp, "generated.cpp")
        with open(cpp_path, "w", encoding="utf-8") as f:
            f.write(cpp_code)
        exe_path = os.path.join(tmp, "a.out")
        try:
            import shutil
            shutil.copyfile(exe, exe_path)
            print(f"Copied executable to: {exe_path}")
        except Exception as e:
            print(f"Error copying executable: {e}")
            exe_path = exe
        # Export results
        csv_path = os.path.join(tmp, "benchmark.csv")
        df.to_csv(csv_path, index=False)
        print(f"Benchmark results saved to: {csv_path}")
        md = df.to_markdown(index=False)
        print("--- Build and bench finished ---")
        return md, cpp_path, exe_path, csv_path, build_log
    except Exception as e:
        error_message = f"An error occurred: {type(e).__name__}: {e}\nTraceback:\n{traceback.format_exc()}"
        print(error_message) # Print to console for debugging
        print("--- Build and bench failed ---")
        return f"Error: {type(e).__name__}: {e}", None, None, None, error_message

import traceback

### 13) Launch the app

In [61]:
with gr.Blocks() as app:
    gr.Markdown("# Python to C++ Converter & Benchmark")
    gr.Markdown("Convert a Python function to C++ using a frontier or open-source LLM, then compile and benchmark.")

    with gr.Row():
        with gr.Column():
            py_code_input = gr.Code(label="Python Function", language="python", value=DEFAULT_PY, lines=10)
            func_name_input = gr.Textbox(label="Python Function Name", value=DEFAULT_FUNC)
            cpp_sig_input = gr.Textbox(label="Required C++ Signature", value=DEFAULT_CPP_SIG)
            perf_hints_input = gr.Textbox(label="Performance Hints (Optional)", value="")
            tests_input = gr.Textbox(label="Test Cases (comma-separated integers)", value=", ".join(map(str, DEFAULT_TESTS)))

            # Conditional visibility for Frontier Model section
            if openai_client or anthropic_client:
                with gr.Accordion("Frontier Model (Optional)", open=False):
                    providers = []
                    if openai_client:
                        providers.append("OpenAI")
                    if anthropic_client:
                        providers.append("Anthropic")
                    default_provider = providers[0] if providers else None
                    default_model = "gpt-4o-mini" if "OpenAI" in providers else "claude-3-5-sonnet-20240620" if "Anthropic" in providers else ""

                    frontier_provider = gr.Radio(providers, label="Provider", value=default_provider)
                    frontier_model = gr.Textbox(label="Model Name", value=default_model)
                    frontier_generate_btn = gr.Button("Generate C++ (Frontier)")
            else:
                 gr.Markdown("To use the Frontier Model option, please set your `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` environment variable and re-run the setup cells.")


            with gr.Accordion("Open-Source Model", open=True):
                os_model_id = gr.Dropdown(OS_CHOICES, label="Model ID", value=OS_DEFAULT)
                os_temperature = gr.Slider(0, 1, value=0.2, label="Temperature")
                os_top_p = gr.Slider(0, 1, value=0.95, label="Top P")
                os_generate_btn = gr.Button("Generate C++ (Open-Source)")

        with gr.Column():
            cpp_code_output = gr.Code(label="Generated C++ Code", language="cpp", lines=15, interactive=True)
            build_bench_btn = gr.Button("Build & Benchmark C++")
            benchmark_output = gr.Markdown("Benchmark Results:")

            with gr.Accordion("Build & Run Details", open=False):
                build_log_output = gr.Textbox(label="Build Log", lines=5)
                cpp_file_link = gr.File(label="Generated C++ Source File")
                exe_file_link = gr.File(label="Compiled Executable")
                csv_file_link = gr.File(label="Benchmark CSV Results")


    # Define interactions
    if openai_client or anthropic_client:
        frontier_generate_btn.click(
            generate_cpp_frontier,
            inputs=[frontier_provider, frontier_model, py_code_input, cpp_sig_input, perf_hints_input],
            outputs=cpp_code_output
        )

    os_generate_btn.click(
        generate_cpp_open_source,
        inputs=[os_model_id, py_code_input, cpp_sig_input, perf_hints_input, os_temperature, os_top_p],
        outputs=cpp_code_output
    )

    build_bench_btn.click(
        do_build_and_bench,
        inputs=[cpp_code_output, py_code_input, func_name_input, tests_input],
        outputs=[benchmark_output, cpp_file_link, exe_file_link, csv_file_link, build_log_output]
    )

In [62]:
# Uncomment to launch in this notebook.
app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://64534fbf96673188ea.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## Notes for Google Colab (T4 GPU)
- Keep the default 7B open‑source model in **4‑bit** for memory headroom.
- If OOM occurs, switch to `microsoft/Phi-3-mini-4k-instruct` (smaller generalist) or reduce `max_new_tokens` in the code.
- For **frontier models**, ensure your `OPENAI_API_KEY` / `ANTHROPIC_API_KEY` are set.
- The default demo `sum_squares(n)` uses integer I/O; for other signatures, adjust the small C++ template (`TEMPLATE`) to parse inputs & print outputs.

In [63]:
from google.colab import userdata

try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Loaded OPENAI_API_KEY from secrets.")
except userdata.SecretNotFoundError:
    print("OPENAI_API_KEY not found in secrets.")

try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("Loaded ANTHROPIC_API_KEY from secrets.")
except userdata.SecretNotFoundError:
    print("ANTHROPIC_API_KEY not found in secrets.")

# Verify they are set in the environment for the next cell
print(f"OPENAI_API_KEY is set: {os.environ.get('OPENAI_API_KEY') is not None}")
print(f"ANTHROPIC_API_KEY is set: {os.environ.get('ANTHROPIC_API_KEY') is not None}")

Loaded OPENAI_API_KEY from secrets.
Loaded ANTHROPIC_API_KEY from secrets.
OPENAI_API_KEY is set: True
ANTHROPIC_API_KEY is set: True
